In [1]:
import os
import matplotlib.pyplot as plt
import xarray as xr
import cartopy.crs as ccrs
from urllib.request import urlretrieve

In [2]:
filename = 'tas_mon_mod_ssp585_192_ave' # SSPs:126, 245, 370, 585 

print(os.path.exists(filename))

if os.path.exists(filename) == False:

    # 如果没有这个文件，自动去网上下载

    url = 'https://climexp.knmi.nl/dyn_links/'+filename

    urlretrieve(url, filename)

False


HTTPError: HTTP Error 404: Not Found

In [ ]:
# 打开这个NC文件

ncid = xr.open_dataset(filename)

# 因为源文件是K，转为摄氏度

ncid['tas'] = ncid['tas'] - 273.15

In [ ]:
# 选取北半球的数据

ncid = ncid.sel(lat = slice(0,90))

In [ ]:
# 选取一部分历史时期(1901-2014)的数据

ncid = ncid.sel(time = slice('1901','2014'))

In [ ]:
# 计算年平均值

ncid = ncid.resample(time = '1YE').mean('time') 
ncid

In [ ]:
# 计算一下1971-2000期间的平均值，然后画个地图看看

fig, axis = plt.subplots(1, 1, subplot_kw=dict(projection=ccrs.Orthographic(0, 90)))

ncid.sel(time = slice('1971','2000')).mean('time')['tas'].plot(transform=ccrs.PlateCarree()),  # this is important!)

axis.coastlines()  # cartopy function

In [ ]:
# 计算一下1871-1900期间的平均值
# 再计算一下2071-2100期间的平均值
# 用2071-2100减去1871-1900，然后画个地图看看两个时期之间的气温差异

t1901 = ncid.sel(time = slice('1901','1914')).mean('time')
t2001 = ncid.sel(time = slice('2001','2014')).mean('time')

fig, axis = plt.subplots(1, 1, subplot_kw=dict(projection=ccrs.Orthographic(0, 90)))

(t2001 - t1901)['tas'].plot(transform=ccrs.PlateCarree()),  # this is important!)

axis.coastlines()  # cartopy function

In [ ]:
# 计算0-30N的平均气温
# 计算60-90N的平均气温
# 比较一下两个时间序列

ncid.sel(lat = slice(0,30)).mean('lat').mean('lon')['tas'].plot(label = '0-30N')
ncid.sel(lat = slice(60,90)).mean('lat').mean('lon')['tas'].plot(label = '60-90N')
plt.legend()

### <span style='color: red;'>Tips</span>: 你会发现，这两个序列差的太远了，不太好比较。
###     为了比较，往往会先计算距平值(Anomaly or Departure)

In [ ]:
anom = ncid - ncid.sel(time = slice('1901','1930')).mean('time')

anom.sel(lat = slice(0,30)).mean('lat').mean('lon')['tas'].plot(label = '0-30N')
anom.sel(lat = slice(60,90)).mean('lat').mean('lon')['tas'].plot(label = '60-90N')

plt.legend()

In [ ]:
# 计算一下1901-1914年期间，不同纬度的平均气温
# 计算一下2001-2014年期间，不同纬度的平均气温

t1901.mean('lon')['tas'].plot(label = '1901-1914')
t2001.mean('lon')['tas'].plot(label = '2001-2014')

plt.legend()

In [ ]:
(t2001 - t1901).mean('lon')['tas'].plot(label = '[1901-1914] to [2001-2014]')

plt.legend()

## <span style='color: red;'>Q</span>: 同学们可以尝试看看用距平值（anom）计算结果如何？